# The paper tables (shortlist) — decodability before editability

Reads each listed run's `scores.json` (written by `master_eval.ipynb`), the floors under `runs/_baselines/`, and two experiment score files (Table 3 alignment, Table 4 Bayes floor). Rendering and selection live in `pim.figures.tables`; this notebook only sets the run lists and calls one function per table. Every table is an image.

Conventions: Othello above discworld with a heavy rule between them; rows follow the lists below; Othello's Edit Index is the **symmetric-difference** construction (union kept in `scores.json`); every per-cell decodability value is that cell's own optimum over residual points; probes are held out by sequence.

* **Table 1** decodability against its floors · **1b/1c** discworld by component · **1d/1e** above the random-init floor
* **Table 2** editability (Edit Index; fidelity ratio) · **2b** best arm per cell · **2c** gridified discworld targets
* **Table 3** true edit direction vs probe row space (± Haufe) · **Table 4** test loss vs Bayes floor · **Table 5** seed replicates
* **Fig 1** training curve (L-oth-20m, L-dw-20m) · **Fig 2** probe-capacity sweep


In [ ]:
# [1] THE RUN LISTS — the only thing to edit. Othello first, then discworld; the order here is the row order.
RUNS_OTH = ['L-oth-20m', 'L-oth-20m-mse', 'L-oth-adjacent-20m', 'L-oth-noflip-20m']
RUNS_DW = ['L-dw-noiseless-20m', 'L-dw-8ray-20m', 'L-dw-blink-20m', 'L-dw-8ray-tok-20m']

import sys
from pathlib import Path

import matplotlib.pyplot as plt

REPO = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO))
from pim.figures import tables as T

# THE BASIS (2026-09-15, Sevan): every discworld number in these tables — decodability, floors, PI / GS / IM —
# comes from this regression basis and none from the other. "frustum" (canonical) or "cartesian" (every run
# carries both blocks). A run without the requested block (dw-8ray-obs5: cartesian only) shows in the block it
# has, marked * when frustum is requested. Grid-defined targets are the same in either basis.
BASIS = 'frustum'
T.set_basis(BASIS)

# Seed replicates (<run>__seed<k>) fold into ± columns. Guard: pooled only at a MATCHED training budget
# (±10%); set pool_budgets=True to pool every budget of a run (Table 5 then says 'mixed budgets').
POOL_BUDGETS = False
F = T.collect(RUNS_OTH, RUNS_DW, pool_budgets=POOL_BUDGETS, label='paper')
print(f'{len(F.df)} (run, target) rows · frame-set runs {sorted(F.frame_set) or "none"} · '
      f'replicate cells {len(F.rep_sd)}' + (f' · MISSING {F.missing}' if F.missing else ''))

In [ ]:
# [2] TABLE 1 — decodability against its floors (right-aligned observation; random-init of the listed architectures)
fig = T.table_decodability(F, '1')
plt.show() if fig else print('Table 1: nothing to draw')

In [ ]:
# [3] TABLES 1b / 1c — discworld decodability by component, each cell at its own best point
for fig in T.tables_components(F):
    plt.show()

In [ ]:
# [4] TABLES 1d / 1e — the same, minus the random-init floor's own per-cell optimum
for fig in T.tables_components(F, above_floor=True):
    plt.show()

In [ ]:
# [5] TABLE 2 — editability (a) Edit Index with the unedited floor, (b) fidelity ratio; TABLE 2b the arms
fig = T.table_editability(F, '2')
plt.show() if fig else print('Table 2: nothing to draw')
fig = T.table_arms(F, '2b')
plt.show() if fig else None

In [ ]:
# [6] TABLE 2c — the gridified discworld targets only, coarse → fine
fig = T.table_gridified(F, '2c')
plt.show() if fig else print('Table 2c: no gridified targets among the listed runs')

In [ ]:
# [7] TABLE 3 — true edit direction vs the probe row space at the best PI point, before / after Haufe;
#     PI after Haufe is filled by the queued Haufe-edit run (blank until then)
fig = T.table_alignment(F, '3')
plt.show() if fig else print('Table 3: nothing to draw')

In [ ]:
# [8] TABLE 4 — test loss vs the estimated Bayes floor (experiments/bayes_floor/scores/test_loss.json)
fig = T.table_bayes(F, '4')
plt.show() if fig else print('Table 4: nothing to draw')

In [ ]:
# [9] TABLE 5 — seed replicates (mean ± SD per run and target)
fig = T.table_seed_variance(F, '5')
plt.show() if fig else print('Table 5: no seed replicates among the listed runs')

In [ ]:
# [10] FIG 1 — training curve for the two canonical runs (Othello on top)
fig = T.fig_training_curve(['L-oth-20m', 'L-dw-20m'], '1')
plt.show() if fig else print('Fig 1: no training-curve checkpoints')

In [ ]:
# [11] FIG 2 — the probe-capacity sweep
fig = T.fig_capacity('2')
plt.show() if fig else print('Fig 2: no capacity sweep scores')